# SQL Agent: Natural Language to SQL
## From English to SQL — Natural Language Queries on a Live Database
⏱ ~12 min

A **SQL agent** lets you ask questions in plain English and get answers from a database — no SQL knowledge required from the user. Under the hood it uses the **ReAct** loop: the LLM reasons about what to do, calls a tool (`list_tables`, `describe_table`, `run_sql`), reads the result, and repeats until it has a final answer.

By the end of this workbook you will have:
- Built a local SQLite database and seeded it with sales data
- Written three SQL tools from scratch with safety guards
- Created a ReAct agent with LangGraph's `create_react_agent`
- Queried the agent in plain English and watched it reason step-by-step
- Extended the agent with a custom aggregation tool

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Database Setup** — SQLite, schema, seed data |
| 2 | **SQL Tools** — list, describe, run with safety guard |
| 3 | **ReAct Agent** — wiring LLM + tools into a loop |
| 4 | **Live Queries** — watch the agent reason step-by-step |
| 5 | **Safety & Limitations** — what can go wrong and why |
| 6 | **Custom Tools** — extending the agent's capabilities |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Gao, Y., et al. (2023). *Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation.* https://arxiv.org/abs/2308.15363  
> Yao, S., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> LangGraph `create_react_agent` docs: https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.chat_agent_executor.create_react_agent  
> LangChain SQL toolkit reference: https://python.langchain.com/docs/integrations/tools/sql_database/

In [ ]:
# Install dependencies -- runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain-openai",
            "langgraph",
            "langchain-core",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected -- skipping install.")

In [ ]:
import os
import sqlite3
from pathlib import Path

# -- API key -------------------------------------------------------------------
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# -- Sanity check -------------------------------------------------------------
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed. "
    "Local: add it to .env  |  Colab: Secrets panel -> Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True  (starts with {key[:7]}...)")

---

## How the SQL Agent Works

### Text-to-SQL Approaches

There are several ways to let users query databases in natural language:

| Approach | How it works | Pros | Cons |
|----------|-------------|------|------|
| **Prompt engineering** | Stuff the full schema into a single prompt, ask for SQL | Simple, one LLM call | Fragile with large schemas, no iteration |
| **Fine-tuning** | Train a model on (question, SQL) pairs | High accuracy on known queries | Expensive, needs labeled data, can't adapt |
| **ReAct agent** (this workshop) | LLM discovers schema via tools, writes SQL, retries on error | Adaptive, handles unknown schemas | More LLM calls, less predictable |
| **Semantic layer** | Pre-defined metrics + NL mapping | Consistent, auditable | Heavy setup, limited flexibility |

The **ReAct agent** approach is the most flexible: the LLM explores the database like a developer would -- first checking what tables exist, then reading the schema, then writing and running a query.

---

### The ReAct Loop -- ASCII Diagram

```
User: "Which product had the highest total sales?"
         |
         v
  +----------------------------------+
  |  THOUGHT: I need to check what  |
  |  tables exist first.            |  <- LLM reasons
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  ACTION: list_tables()          |  <- LLM picks a tool
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  OBSERVATION: "sales"           |  <- tool result fed back
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  THOUGHT: I should check the    |
  |  schema of the sales table.     |
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  ACTION: describe_table(sales)  |
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  OBSERVATION: id, product,      |
  |  region, amount, date           |
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  ACTION: run_sql(               |
  |    SELECT product,              |
  |    SUM(amount) AS total         |
  |    FROM sales                   |
  |    GROUP BY product             |
  |    ORDER BY total DESC LIMIT 1) |
  +--------------+-------------------+
                 |
                 v
  +----------------------------------+
  |  OBSERVATION: Widget A, 4290.00 |
  +--------------+-------------------+
                 |
                 v
  "Widget A had the highest total sales at $4,290."
```

> **Source**: ReAct pattern from Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629

---

## Part 1 -- Database Setup

---

We use **SQLite** -- it runs entirely in a file, requires no server, and is ideal for local development and teaching. The database has one table (`sales`) with 8 rows covering three products, four regions, and four months.

### Why SQLite for a SQL agent?

| Property | SQLite | Postgres / MySQL |
|----------|--------|------------------|
| Server needed | No -- file on disk | Yes |
| Python stdlib | Yes (`import sqlite3`) | No (needs driver) |
| Good for prototyping | Yes | Yes |
| Good for production | Rarely | Yes |
| Read-only connection | Via URI (`?mode=ro`) | Via role permissions |

The agent pattern you learn here works unchanged with Postgres or MySQL -- just swap the connection string.

In [ ]:
# --- 1-A: Create and seed the database ---------------------------------------

DB_PATH = "sales.db"


def seed_db(db_path: str = DB_PATH) -> None:
    """Create the sales table and insert sample rows (idempotent)."""
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS sales (
            id      INTEGER PRIMARY KEY,
            product TEXT,
            region  TEXT,
            amount  REAL,
            date    TEXT
        );
        INSERT OR IGNORE INTO sales VALUES
            (1, 'Widget A', 'North',  1200.00, '2024-01-15'),
            (2, 'Widget B', 'South',   850.50, '2024-01-20'),
            (3, 'Widget A', 'East',   2100.00, '2024-02-05'),
            (4, 'Widget C', 'North',   430.00, '2024-02-18'),
            (5, 'Widget B', 'West',   1750.00, '2024-03-01'),
            (6, 'Widget A', 'South',   990.00, '2024-03-22'),
            (7, 'Widget C', 'East',    610.00, '2024-04-10'),
            (8, 'Widget B', 'North',  1340.00, '2024-04-28');
    """)
    conn.commit()
    conn.close()


seed_db()
print(f"Database ready: {DB_PATH}")

In [ ]:
# --- 1-B: Inspect the raw data -----------------------------------------------
# Always look at your data before building an agent over it.
# This is what the agent will be querying.

conn = sqlite3.connect(DB_PATH)
rows = conn.execute("SELECT * FROM sales").fetchall()
conn.close()

header = f"{'id':>3}  {'product':<10} {'region':<8} {'amount':>8}  {'date'}"
print(header)
print("-" * len(header))
for r in rows:
    print(f"{r[0]:>3}  {r[1]:<10} {r[2]:<8} {r[3]:>8.2f}  {r[4]}")

total = sum(r[3] for r in rows)
print("-" * len(header))
print(f"{'':3}  {'':10} {'TOTAL':>8} {total:>8.2f}")

In [ ]:
# --- 1-C: Ground-truth summary stats (plain SQL, no agent) -------------------
# Compute expected answers now so you can verify the agent later.

conn = sqlite3.connect(DB_PATH)

print("Sales by product:")
for row in conn.execute(
    "SELECT product, COUNT(*) as n, SUM(amount) as total "
    "FROM sales GROUP BY product ORDER BY total DESC"
).fetchall():
    print(f"  {row[0]:<10} {row[1]} sales  ${row[2]:,.2f}")

print("\nSales by region:")
for row in conn.execute(
    "SELECT region, COUNT(*) as n, SUM(amount) as total "
    "FROM sales GROUP BY region ORDER BY total DESC"
).fetchall():
    print(f"  {row[0]:<8} {row[1]} sales  ${row[2]:,.2f}")

conn.close()

print("\n^ These are the ground-truth answers. Verify the agent matches them later.")

---

## Part 2 -- SQL Tools

---

The agent needs tools to explore and query the database. We build three, from broad to specific:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `list_tables` | Returns all table names | First -- discover what exists |
| `describe_table` | Returns column names and types | Second -- understand the schema |
| `run_sql` | Executes a SELECT query | Third -- get the actual data |

Each tool is decorated with `@tool`. The **docstring is critical** -- the LLM reads it to decide when and how to call each tool. A vague docstring produces worse tool selection.

### Safety Considerations for SQL Agents

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Write operations** | Agent generates DELETE/DROP if misled | Block DDL/DML keywords in tool |
| **Full table scans** | SELECT * on a 10M-row table | Row limit via `fetchmany()` |
| **SQL injection via prompt** | Adversarial input crafts malicious SQL | Read-only DB user + keyword guard |
| **Schema exposure** | Agent leaks column names in answers | Restrict schema access per user role |
| **Runaway queries** | Long-running aggregation blocks DB | Query timeout at connection level |

In [ ]:
# --- 2-A: list_tables and describe_table -------------------------------------
from langchain_core.tools import tool


# @tool wraps the function and uses its docstring as the tool description.
# The LLM picks which tool to call based solely on those docstrings.
@tool
def list_tables() -> str:
    """List all tables in the database. Call this first to discover available data."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()
    conn.close()
    return ", ".join(r[0] for r in rows) or "No tables found."


# PRAGMA table_info returns cid, name, type, notnull, dflt_value, pk.
# Returning just name+type keeps the context window lean for the agent.
@tool
def describe_table(table: str) -> str:
    """Return the column schema of a table (column name and data type).
    Call this after list_tables to understand the structure before writing SQL."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    conn.close()
    return "\n".join(f"{r[1]} ({r[2]})" for r in rows)


# Sanity check
print("Tables   :", list_tables.invoke({}))
print("\nSchema for 'sales':")
print(describe_table.invoke({"table": "sales"}))

In [ ]:
# --- 2-B: run_sql with safety guard ------------------------------------------

# Keyword blocklist is intentionally coarse — a proper guard would parse the
# AST instead. For production, prefer a read-only DB connection or RBAC.
FORBIDDEN_KEYWORDS = {"INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE"}


@tool
def run_sql(query: str) -> str:
    """Execute a read-only SQL SELECT query against the database.
    Returns up to 20 rows as comma-separated text.
    Only SELECT statements are permitted -- write operations are blocked."""
    upper = query.upper()
    if any(kw in upper for kw in FORBIDDEN_KEYWORDS):
        return "Error: Only SELECT queries are permitted."
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.execute(query)
# fetchmany(20) caps rows returned per call, preventing runaway full-table scans.
# The agent can page by re-calling with LIMIT/OFFSET if it needs more.
        rows = cur.fetchmany(20)
        headers = [d[0] for d in cur.description]
        lines = [", ".join(headers)] + [
            ", ".join(str(v) for v in row) for row in rows
        ]
        return "\n".join(lines)
    except sqlite3.Error as e:
        return f"SQL error: {e}"
    finally:
        conn.close()


# Test 1: safety guard
print("Safety guard test:")
print(run_sql.invoke({"query": "DROP TABLE sales"}))

# Test 2: valid query
print("\nTop 3 sales by amount:")
print(run_sql.invoke({
    "query": "SELECT product, region, amount FROM sales ORDER BY amount DESC LIMIT 3"
}))

In [ ]:
# --- 2-C: Why docstrings drive agent behaviour --------------------------------
# The LLM sees the docstring as a description of what the tool does and when to
# call it. Print the schema to see exactly what the agent receives.

for t in [list_tables, describe_table, run_sql]:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description}")
    schema = t.args_schema.schema() if t.args_schema else {}
    props = schema.get("properties", {})
    if props:
        for param, detail in props.items():
            print(f"  Param: {param} -- {detail.get('description', detail.get('type', ''))}")
    else:
        print("  Params: none")
    print()

---

## Part 3 -- Building the ReAct Agent

---

`create_react_agent` from LangGraph wires the LLM and tools together into the ReAct loop automatically. You supply the model and a list of tools -- the framework handles the message routing, tool dispatch, and observation injection.

### What `create_react_agent` does internally

```
LLM response
     |
     +-- has tool_calls? --> dispatch to tool --> append ToolMessage --> loop
     |
     +-- no tool_calls?  --> return AIMessage as final answer
```

The LangGraph state is a simple dict with a `messages` list. Each turn appends to it, so the full conversation history (including every tool call and result) is always in context.

### Message types in the ReAct state

| Message type | From | Contains |
|-------------|------|----------|
| `HumanMessage` | User | The original question |
| `AIMessage` | LLM | Either `tool_calls` (intermediate) or the final answer |
| `ToolMessage` | Tool | The raw result string from the tool function |

In [ ]:
# --- 3-A: Create the agent ---------------------------------------------------
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# gpt-4o-mini is fast and cheap; swap for gpt-4o when SQL reasoning fails.
llm = ChatOpenAI(model="gpt-4o-mini")  # fast, cheap, strong at SQL generation
tools = [list_tables, describe_table, run_sql]

# create_react_agent: Reason → Act → Observe loop. No custom graph needed;
# LangGraph's prebuilt node handles tool dispatch and message accumulation.
app = create_react_agent(llm, tools=tools)
print("Agent ready.")
print(f"Tools registered: {[t.name for t in tools]}")

In [ ]:
# --- 3-B: Helper that shows the ReAct trace ----------------------------------
# Printing intermediate tool calls reveals exactly how the agent reasons.
# This is the most important debugging tool in this workshop.


def ask(question: str, agent=None, show_steps: bool = True) -> str:
    """Run a question through the agent and print the tool-call trace."""
    _app = agent or app
    # invoke() runs the full ReAct loop synchronously and returns all messages.
    # Use astream_events() instead if you want token-by-token streaming output.
    result = _app.invoke({"messages": [HumanMessage(question)]})

    if show_steps:
        print(f"Question: {question}")
        print("-" * 60)
        for msg in result["messages"][1:]:  # skip the original HumanMessage
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    args_str = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                    print(f"  [call]   {tc['name']}({args_str if args_str else 'no args'})")
            elif msg.type == "tool":
                preview = msg.content[:120].replace("\n", " / ")
                print(f"  [result] {preview}")
        print("-" * 60)

    answer = result["messages"][-1].content
    print(f"Answer: {answer}\n")
    return answer


print("ask() helper defined.")

---

## Part 4 -- Live Queries

---

Watch the `[call]` and `[result]` lines -- that is the ReAct loop made visible.
The agent discovers the schema on its own, writes SQL, reads the result, and composes its answer in plain English.

**What to observe:**
- Does the agent always call `list_tables` first, or does it skip ahead on repeat questions?
- Does it retry when a first query returns an error?
- How does it translate business concepts ("Q1", "best month") into SQL predicates?

In [ ]:
# --- 4-A: Schema discovery ---------------------------------------------------
# First question: agent starts with no knowledge of the database.
ask("What tables are in the database and what columns do they have?")

In [ ]:
# --- 4-B: Aggregation query --------------------------------------------------
# The agent writes GROUP BY + SUM without being told to.
# Compare its answer to the ground truth from cell 1-C.
ask("Which product had the highest total sales?")

In [ ]:
# --- 4-C: Multi-dimensional query --------------------------------------------
# The agent must GROUP BY region and sort the output.
ask("What was the total sales amount for each region, sorted highest to lowest?")

In [ ]:
# --- 4-D: Business concept translation ---------------------------------------
# The agent must translate "Q1 2024" into a date range.
# No hardcoded mapping -- the LLM knows this from training.
ask("Which sales happened in Q1 2024?")

---

## Part 5 -- Safety, Limitations, and Query Validation

---

SQL agents are powerful but come with failure modes that don't exist in traditional software.

### Query Validation Approaches

| Method | How | Catches |
|--------|-----|--------|
| **Keyword blocklist** (our approach) | String scan before execution | Basic DDL/DML |
| **SQL parser** (e.g. `sqlglot`) | Parse the AST, check statement type | Any statement type, reliably |
| **Read-only DB user** | Database-level permission | All write ops, even if blocklist fails |
| **Query allowlist** | Only pre-approved query patterns allowed | Everything not on the list |
| **LLM pre-check** | Ask the LLM if the query is safe | Semantic safety (ambiguous intent) |

**Best practice**: layer at least two of these. The keyword blocklist alone is bypassable via SQL comments, unicode tricks, or multi-statement queries.

### Known Limitations of ReAct SQL Agents

| Limitation | Why | Workaround |
|-----------|-----|------------|
| Hallucinates column names | LLM may skip `describe_table` | Make docstring say "always call describe_table first" |
| Non-deterministic SQL | Same question can produce different queries | Log queries; add regression tests |
| Fails on complex JOINs | More reasoning steps = more errors | Pre-join into a view; add a join tool |
| Context grows with turns | Every tool result appends to history | Use `recursion_limit` in LangGraph config |

In [ ]:
# --- 5-A: Test the safety guard ----------------------------------------------

print("Blocked queries:")
blocked_attempts = [
    "DROP TABLE sales",
    "DELETE FROM sales WHERE id=1",
    "INSERT INTO sales VALUES (9, 'Widget D', 'East', 999, '2024-05-01')",
    "ALTER TABLE sales ADD COLUMN fake TEXT",
]
for q in blocked_attempts:
    result = run_sql.invoke({"query": q})
    status = "BLOCKED" if "Error" in result else "ALLOWED"
    print(f"  [{status}] {q[:55]}")

In [ ]:
# --- 5-B: Row limit in action ------------------------------------------------
# fetchmany(20) prevents full-table scans from flooding the context window.

conn = sqlite3.connect(DB_PATH)
conn.execute("""
    CREATE TABLE IF NOT EXISTS big_table (
        id INTEGER PRIMARY KEY,
        val TEXT
    )
""")
conn.executemany(
    "INSERT OR IGNORE INTO big_table VALUES (?, ?)",
    [(i, f"row_{i}") for i in range(1, 101)]
)
conn.commit()
conn.close()

result = run_sql.invoke({"query": "SELECT * FROM big_table"})
row_count = len(result.strip().split("\n")) - 1  # subtract header line
print(f"Table has 100 rows -- tool returned {row_count} rows (cap: 20)")

---

## Part 6 -- Custom Tools

---

The agent's behaviour changes when you add well-named, focused tools. A purpose-built `average_by` tool gives the agent a direct path to a common aggregation -- it can reach for the tool instead of writing a GROUP BY query from scratch.

**General principle**: tool names and docstrings act as a vocabulary the LLM reasons with. More specific tools produce more consistent agent behaviour than prompt engineering alone.

In [ ]:
# --- 6-A: Custom aggregation tool -------------------------------------------

# A purpose-built tool trades generality for reliability: the agent can't
# mis-write AVG() syntax when the SQL is baked in. Docstring still matters.
@tool
def average_by(column: str) -> str:
    """Return the average sale amount grouped by the specified column.
    Valid column values: 'product' or 'region'.
    Results are sorted by average amount descending."""
    allowed = {"product", "region"}
    if column not in allowed:
    # Allowlist prevents SQL injection via the column parameter.
        return f"Error: column must be one of {allowed}"
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        f"SELECT {column}, ROUND(AVG(amount), 2) AS avg_amount "
        f"FROM sales GROUP BY {column} ORDER BY avg_amount DESC"
    ).fetchall()
    conn.close()
    return "\n".join(f"{r[0]}: ${r[1]:.2f}" for r in rows)


# Test the tool directly before wiring it to an agent
print(average_by.invoke({"column": "region"}))

In [ ]:
# --- 6-B: Extended agent vs base agent ---------------------------------------
# Does the extended agent reach for average_by instead of writing GROUP BY + AVG?

# More tools = richer vocabulary for the agent, but also more decision surface.
# Watch whether the agent prefers average_by or falls back to GROUP BY + AVG.
app_extended = create_react_agent(
    llm,
    tools=[list_tables, describe_table, run_sql, average_by]
)

question = "What is the average sale amount by region?"

print("=== BASE AGENT (3 tools) ===")
ask(question, agent=app)

print("=== EXTENDED AGENT (4 tools, includes average_by) ===")
ask(question, agent=app_extended)

---

## Part 7 -- Exercises

---

Attempt these before scrolling to the answer key. Each exercise has a starter cell -- fill in the `TODO` and run it.

In [ ]:
# -- Exercise 1 -- Sales count per product ------------------------------------
# Ask the agent: how many individual sales were recorded per product?
# You want a COUNT of rows, not a SUM of amounts.
# Expected: Widget A = 3, Widget B = 3, Widget C = 2

# TODO: write the ask() call


In [ ]:
# -- Exercise 2 -- Best month -------------------------------------------------
# Ask the agent: which month had the highest total sales?
# The date column is stored as 'YYYY-MM-DD'.
# Hint: the agent must extract the month from the date string.
# Verify the answer against cell 1-C.

# TODO: write the ask() call


In [ ]:
# -- Exercise 3 -- Custom tool: top_n_sales -----------------------------------
# Define a new @tool called `top_n_sales` that takes an integer n and returns
# the top n sales rows sorted by amount descending.
# Create a new agent that includes it, then ask:
# "Show me the top 3 individual sales."

# TODO: define the @tool
# TODO: create new agent with [list_tables, describe_table, run_sql, top_n_sales]
# TODO: run the question


---
---

## Answer Key

Review these only after attempting the exercises yourself.

### Exercise 1 -- Sales count per product

The key distinction: "how many" maps to `COUNT(*)`, not `SUM(amount)`. The agent should call `list_tables` -> `describe_table` -> `run_sql` with a `GROUP BY product` + `COUNT(*)` query.

In [ ]:
ask("How many individual sales were recorded per product?")

The agent typically generates:
```sql
SELECT product, COUNT(*) AS sales_count
FROM sales
GROUP BY product
ORDER BY sales_count DESC
```
Result: Widget A = 3, Widget B = 3, Widget C = 2.

### Exercise 2 -- Best month

The agent must extract the month from `YYYY-MM-DD` text. In SQLite, `strftime('%Y-%m', date)` or `substr(date, 1, 7)` both work. The LLM knows this from training on SQLite documentation.

In [ ]:
ask("Which month had the highest total sales?")

The agent typically generates:
```sql
SELECT strftime('%Y-%m', date) AS month, SUM(amount) AS total
FROM sales
GROUP BY month
ORDER BY total DESC
LIMIT 1
```
Result: 2024-03 ($2,740.00) -- Widget B West ($1,750) + Widget A South ($990).

> **Why this works:** the LLM knows SQLite date functions from training. The agent translates a business concept ("best month") into the correct SQL date function without any hardcoded hint.

### Exercise 3 -- Custom tool: `top_n_sales`

A purpose-built tool gives the agent a direct path to a common query. Notice how tool naming shapes which tool the agent reaches for -- `top_n_sales` is more specific than `run_sql` for this task, and the agent will prefer it when available.

In [ ]:
@tool
def top_n_sales(n: int) -> str:
    """Return the top n individual sales rows sorted by amount descending.
    Use this when the user asks for the biggest, largest, or highest individual sales.
    n must be between 1 and 20."""
    if not 1 <= n <= 20:
        return "Error: n must be between 1 and 20."
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        f"SELECT id, product, region, amount, date "
        f"FROM sales ORDER BY amount DESC LIMIT {n}"
    ).fetchall()
    conn.close()
    lines = ["id, product, region, amount, date"] + [
        f"{r[0]}, {r[1]}, {r[2]}, ${r[3]:.2f}, {r[4]}" for r in rows
    ]
    return "\n".join(lines)


app_top_n = create_react_agent(
    llm,
    tools=[list_tables, describe_table, run_sql, top_n_sales]
)

ask("Show me the top 3 individual sales.", agent=app_top_n)

print("> Notice: with top_n_sales available, the agent may reach for it")
print("> directly rather than writing an ORDER BY + LIMIT query from scratch.")

---

## What's Next?

You now have a complete SQL agent pipeline. Here's where to go deeper:

### Improve the agent
- **System prompt**: pass `state_modifier` to `create_react_agent` to restrict the agent to specific tables, or to always explain its SQL before running it
- **Error retry**: catch `SQL error:` in tool results and prompt the agent to fix its query -- LangGraph's interrupt hooks can structure this
- **Multi-table schemas**: add a `customers` or `products` table and watch the agent learn to JOIN across tables

### Harden for production
- **Read-only DB user**: connect via `sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)` for defence-in-depth
- **SQL parser**: replace the keyword blocklist with `sqlglot.parse_one(query)` -- check `statement.key == 'select'`
- **Query timeout**: wrap `conn.execute()` in a thread with `threading.Timer` to prevent runaway scans
- **Observability**: enable LangSmith tracing to see every ReAct step in a dashboard

### Go further
- **Example 15** -- [CrewAI Research Crew](../15-crewai-research-crew/README.md): multi-agent orchestration with a different framework
- **LangGraph docs**: https://langchain-ai.github.io/langgraph/reference/prebuilt/
- **Swap SQLite for Postgres**: use `psycopg2` with a read-only role -- the tool code changes by roughly 3 lines

### Key takeaways

| Concept | What to remember |
|---------|------------------|
| ReAct loop | Reason -> Act -> Observe, repeated until done |
| `create_react_agent` | Wires LLM + tools into the loop; you supply the pieces |
| Tool docstrings | The LLM reads them -- write clear, specific descriptions |
| Safety guard | Block writes at the tool level, not in the prompt |
| Schema discovery | The agent finds columns itself -- no hardcoded SQL needed |
| Custom tools | Specific tools shape agent behaviour more reliably than prompting |

---

### Academic References

> Gao, Y., Liu, Z., et al. (2023). *Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation.* https://arxiv.org/abs/2308.15363  
> Yao, S., Zhao, J., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629  
> Zhuo, T. Y., et al. (2023). *Robustness of LLM-based Text-to-SQL.* https://arxiv.org/abs/2306.07621  
> LangGraph prebuilt agents: https://langchain-ai.github.io/langgraph/reference/prebuilt/

---

*Workshop complete. The next step is Example 15 -- multi-agent orchestration with CrewAI.*